In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import geopandas as gpd
import seaborn as sns
import networkx as nx
from itertools import combinations

In [ ]:
# weather regime color schemes
wr_color_dict1 = dict({
            
            "AR": "#999933", #153,153,51.
            "AT": "#882255",  #136,34,85.
            "EuBL": "#44aa99", #68,170,153.
            "GL": "#332288", #51,34,136.
            "ScBL": "#117733", #17,119,51.
            "ScTr": "#ddcc77", #221,204,119.
            "ZO": "#cc6677", #204,102,119.
            "no": "#808080" #128,128,128.
        
        }) #full color-blind friendly

wr_color_dict2 = dict({
    
        "AR": "#ffd700", #255,215,0
        "AT": "#4b0082",  #75,0,130
        "EuBL": "#44aa99", #68,170,153
        "GL": "#0000ff", #0,0,255
        "ScBL": "#117733", #17,119,51
        "ScTr": "#ddcc77", #221,204,119
        "ZO": "#cc6677", #204,102,119
        "no": "#808080" #128,128,128.
    
}) #color-blind friendly close to the original

wr_color_dict = wr_color_dict2

# Load Fire Observations

In [ ]:
fire_obs = gpd.read_file('/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region.shp')
fire_obs["start_date"] = pd.to_datetime(fire_obs["start_date"])

In [ ]:
fire_obs.columns

# Load Weather Regime

In [ ]:
wr = pd.read_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Weather_Regime_Dependency/wr_daily_classification_2001-2020.csv")
wr["time"] = pd.to_datetime(wr["time"])

In [ ]:
wr_lc = pd.read_csv("/net/rain/hyclimm/data/public/climate/weather_regimes/wr/LC_daily.csv")
wr_lc['date'] = pd.to_datetime(wr_lc["time"].str[:8], format = "%Y%m%d")

In [ ]:
wr_freq = []
init_yr = pd.date_range(start = "2001-01-01", end = "2001-12-31")

for d in init_yr:
   
    center_dates = [d + pd.DateOffset(years=i) for i in range(20)]

    all_dates = pd.DatetimeIndex([])

    for center in center_dates:
        all_dates = all_dates.append(pd.date_range(start = center-pd.Timedelta(days = 45), end = center+pd.Timedelta(days = 45)))

    wr_lc_slice = wr_lc[wr_lc["date"].isin(all_dates)]
    
    freq = {
        'date': d.strftime("%m-%d"),
        "AT": len(wr_lc_slice[wr_lc_slice['wrname'] == "AT"])/len(wr_lc_slice),
        "ScTr": len(wr_lc_slice[wr_lc_slice['wrname'] == "ScTr"])/len(wr_lc_slice),
        "ZO": len(wr_lc_slice[wr_lc_slice['wrname'] == "ZO"])/len(wr_lc_slice),
        "EuBL": len(wr_lc_slice[wr_lc_slice['wrname'] == "EuBL"])/len(wr_lc_slice),
        "ScBL": len(wr_lc_slice[wr_lc_slice['wrname'] == "ScBL"])/len(wr_lc_slice),
        "AR": len(wr_lc_slice[wr_lc_slice['wrname'] == "AR"])/len(wr_lc_slice),
        "GL": len(wr_lc_slice[wr_lc_slice['wrname'] == "GL"])/len(wr_lc_slice),
        "no": len(wr_lc_slice[wr_lc_slice['wrname'] == "no"])/len(wr_lc_slice)
    }
    wr_freq.append(freq)
    
wr_freq = pd.DataFrame(wr_freq)

#re-arrange order to keep consistency with Dominik's paper
wr_freq = wr_freq[['date', 'AT', 'ZO', 'ScTr', 'AR', 'EuBL', 'ScBL', 'GL', 'no']]

In [ ]:
#save
#wr_freq.to_csv('/net/rain/hyclimm/data/projects/SynFire/WP1/Weather_Regime_Dependency/weather_regime_fraction_2001-2020.csv', index = False)

In [ ]:
#plot frequency
#----------------------------------------
fig, ax = plt.subplots(figsize = (10, 6))

wr_freq.plot(kind = 'bar', stacked = True, ax = ax, color = wr_color_dict, width = 1)

ax.set_ylabel("Weather Regime Fraction (-)", fontsize = 16)
ax.set_xlabel("Calendar Day", fontsize = 16)
ax.legend(loc = "lower center", ncol = 8, bbox_to_anchor = (0.5, -0.35), frameon = False, fontsize = 15, handletextpad = 0.1, columnspacing = 1)
ax.set_ylim(0,1)
ax.set_xticks([i for i, date in enumerate(init_yr) if date.is_month_start])
ax.set_xticklabels(['01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01', '10-01','11-01', '12-01'], rotation = 45)

for i, date in enumerate(init_yr):
     if date.is_month_start and i != 0:
        ax.axvline(i, color = 'black', linewidth = 0.2)
         
ax.tick_params(axis = "both", labelsize = 13)

plt.savefig("/home/lixinh/SynFire_Scripts/WP1/Figures/S15_weather_regime_frequency.png", dpi = 600, bbox_inches = "tight")
plt.show()

# Make SynFire 7d dataframe

In [ ]:
# definitions of "type" (Based on a 7-day left-aligned sliding window)
# no: no fire occurred in all regions
# reg: fire(s) occurred within any single region
# low: fires co-occurred in 2-3 regions
# medium: fires co-occurred in 4-5 regions
# high: fires co-occurred in more than 5 regions

synfire_7d = pd.DataFrame(columns = ["date", "regsyn", "consyn", "WR", "BI", "SC", "ME", "NEA", "FR", "AL", "SEA", "IP", "WMD", "EMD", "id_list", "WR_list", "type"])
synfire_7d["date"] = pd.date_range(start = "2001-01-01", end = "2020-12-25")

In [ ]:
synfire_7d

In [ ]:
for ind, row in synfire_7d.iterrows():
    
    t = row["date"]

    # WR
    wr_7d = list(wr[(wr["time"] >= t) & (wr["time"] <= t + pd.Timedelta(days=6))]["wrname"])
    synfire_7d.at[ind, "WR_list"] = wr_7d

    #identify the most frequent WR, if not unique, assign all 
    wr_dict = dict(pd.Series(wr_7d).value_counts())
    max_freq = max(wr_dict.values())
    max_wr = [wr for wr, freq in wr_dict.items() if freq == max_freq]
    synfire_7d.at[ind, "WR"] = max_wr

    # filter fire obs
    fire_obs_7d = fire_obs[(fire_obs["start_date"] >= t) & (fire_obs["start_date"] <= t + pd.Timedelta(days = 6))]

    
    #-------------------------------------------
    if len(fire_obs_7d) <= 1: #no synchronicity
        
        synfire_7d.loc[ind, "regsyn"] = 0
        synfire_7d.loc[ind, "consyn"] = 0

        if len(fire_obs_7d) == 0:  #no fire
            synfire_7d.loc[ind, "type"] = "no"
            
        else: # one fire in any region
            
            synfire_7d.loc[ind, "type"] = "reg"
            
            #record id_list
            synfire_7d.at[ind, "id_list"] = list(fire_obs_7d["ptch_id"])
            reg_unique = fire_obs_7d["region"].item()
            synfire_7d.loc[ind, ["BI", "SC", "ME", "NEA", "FR", "AL", "SEA", "IP", "WMD", "EMD"]] = [1 if reg == reg_unique else 0 for reg in ["BI", "SC", "ME", "NEA", "FR", "AL", "SEA", "IP", "WMD", "EMD"]]
            
    #-------------------------------------------  
    else: #synchronicity
        
        #record id_list
        synfire_7d.at[ind, "id_list"] = list(fire_obs_7d["ptch_id"])
        regs_list = list(fire_obs_7d["region"])
        regs_set = list(set(fire_obs_7d["region"])) #unique list of affected regions

        if len(regs_set) == 1: #regional synchronicity
            synfire_7d.loc[ind, "regsyn"] = 1
            synfire_7d.loc[ind, "consyn"] = 0

            #record the number of fires for the region
            synfire_7d.loc[ind, ["BI", "SC", "ME", "NEA", "FR", "AL", "SEA", "IP", "WMD", "EMD"]] = [regs_list.count(reg) if reg in regs_set else 0 for reg in ["BI", "SC", "ME", "NEA", "FR", "AL", "SEA", "IP", "WMD", "EMD"]]
            synfire_7d.loc[ind, "type"] = "reg"
            
        else: # continental synchronicity
            synfire_7d.loc[ind, "regsyn"] = 0
            synfire_7d.loc[ind, "consyn"] = len(regs_set)

            #record the number of fires for each region
            synfire_7d.loc[ind, ["BI", "SC", "ME", "NEA", "FR", "AL", "SEA", "IP", "WMD", "EMD"]] = [regs_list.count(reg) if reg in regs_set else 0 for reg in ["BI", "SC", "ME", "NEA", "FR", "AL", "SEA", "IP", "WMD", "EMD"]]

            #assign event type
            synfire_7d.loc[ind, "type"] = "low" if len(regs_set) in [2, 3] else "medium" if len(regs_set) in [4, 5] else "high"

In [ ]:
# remove rows with duplicate id_list but keep all the rows with NaN (as this is needed to calculate the dependency of "no" on weather regimes)
synfire_7d_nona = synfire_7d.dropna()
synfire_7d_nona = synfire_7d_nona.drop_duplicates(subset = ["id_list"], keep = "first")
synfire_7d_na = synfire_7d[synfire_7d.isna().any(axis=1)]
synfire_7d_uni_id_list = pd.concat([synfire_7d_nona, synfire_7d_na], ignore_index = True)
synfire_7d_uni_id_list = synfire_7d_uni_id_list.sort_values(by = "date")
synfire_7d_uni_id_list = synfire_7d_uni_id_list.reset_index(drop = True)

In [ ]:
synfire_7d_uni_id_list

In [ ]:
#save
synfire_7d_uni_id_list.to_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Weather_Regime_Dependency/SynFire_7d_unique_id_list_five_levels.csv", index = False)

# Calculate Dependency

In [ ]:
#include no regime

def dependency_calculator(season):

    #----------------------------
    #month index
    mon_ind = [3, 4, 5] if season == "MAM" else [6, 7, 8] if season == "JJA" else [9, 10, 11]
    
    #load wr daily classification
    wr = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Weather_Regime_Dependency/wr_daily_classification_{season}_2001-2020.csv")
    
    #calculate wr frequency
    p_wr = dict(zip(["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL", "no"], [len(wr[wr["wrname"] == wrt])/len(wr) for wrt in ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL", "no"]]))
    
    #load synfire dataset with unique fire patch id list
    synfire_7d = pd.read_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Weather_Regime_Dependency/SynFire_7d_unique_id_list_five_levels.csv")
    
    #subset synfire_7d to each season
    synfire_7d["date"] = pd.to_datetime(synfire_7d["date"])
    synfire_7d = synfire_7d[synfire_7d["date"].dt.month.isin(mon_ind)]
    
    #total number of day
    dates = pd.date_range(start = "2001-01-01", end = "2020-12-31")
    n = len([date for date in dates if date.month in mon_ind]) 
    
    #----------------------------
    #probability of events
    
    #no (no fire within the entire continent)
    p_no = len(synfire_7d[synfire_7d["type"] == "no"])/n
    
    #reg (fire(s) within a single region)
    p_reg = len(synfire_7d[synfire_7d["type"] == "reg"])/n
    
    #lowconsyn (2-3 regions affected)
    p_lowconsyn = len(synfire_7d[synfire_7d["type"] == "low"])/n
    
    #mediumconsyn (4-5 regions affected)
    p_mediumconsyn = len(synfire_7d[synfire_7d["type"] == "medium"])/n 
    
    #highconsyn (more than 5 regions affected)
    p_highconsyn = len(synfire_7d[synfire_7d["type"] == "high"])/n 
    
    #----------------------------
    #initialize dependency table
    dp = pd.DataFrame(columns = ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL", "no"], index = ["no", "reg", "lowconsyn", "mediumconsyn", "highconsyn"])
    
    #calculate dependency
    for wrt in ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL", "no"]:
        
        p_no_wr = len(synfire_7d[(synfire_7d["type"] == "no") & (synfire_7d["WR"].apply(lambda x: wrt in x))])/n
        
        p_reg_wr = len(synfire_7d[(synfire_7d["type"] == "reg")  & (synfire_7d["WR"].apply(lambda x: wrt in x))])/n
        
        p_lowconsyn_wr = len(synfire_7d[(synfire_7d["type"] == "low") & (synfire_7d["WR"].apply(lambda x: wrt in x))])/n
        
        p_mediumconsyn_wr = len(synfire_7d[(synfire_7d["type"] == "medium") & (synfire_7d["WR"].apply(lambda x: wrt in x))])/n
        
        p_highconsyn_wr = len(synfire_7d[(synfire_7d["type"] == "high") & (synfire_7d["WR"].apply(lambda x: wrt in x))])/n

        #-----------------------------------
        dp_no_wr = (p_no_wr/p_wr[wrt])/p_no
        
        dp_reg_wr = (p_reg_wr/p_wr[wrt])/p_reg
        
        dp_lowconsyn_wr = (p_lowconsyn_wr/p_wr[wrt])/p_lowconsyn
        
        dp_mediumconsyn_wr = (p_mediumconsyn_wr/p_wr[wrt])/p_mediumconsyn
        
        dp_highconsyn_wr = (p_highconsyn_wr/p_wr[wrt])/p_highconsyn

        #-----------------------------------
        dp[wrt] = [dp_no_wr, dp_reg_wr, dp_lowconsyn_wr, dp_mediumconsyn_wr, dp_highconsyn_wr]
    
    #----------------------------
    #record sample size
    dp["n"] = [len(synfire_7d[synfire_7d["type"] == "no"]), #no
               len(synfire_7d[synfire_7d["type"] == "reg"]), #regional fire(s)
               len(synfire_7d[synfire_7d["type"] == "low"]), #low
               len(synfire_7d[synfire_7d["type"] == "medium"]), #medium
               len(synfire_7d[synfire_7d["type"] == "high"]) #high
              ]

    
    dp.to_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Weather_Regime_Dependency/Weather_Regime_Dependency_five_levels_{season}_incl_noregime.csv", index = True, index_label = "synlevel")

In [ ]:
dependency_calculator("MAM")
dependency_calculator("JJA")
dependency_calculator("SON")

# Plot (include no regime)

In [ ]:
wr = plt.figure(figsize=(15, 15), constrained_layout=True)

gs_out = wr.add_gridspec(2, 1, height_ratios=(1, 1), hspace = 0.05) 
upper = wr.add_subfigure(gs_out[0, 0])  
lower = wr.add_subfigure(gs_out[1, 0])  
gs_in = lower.add_gridspec(3, 2, width_ratios=(1, 1), height_ratios = (1, 1, 1))
rel_conpro = lower.add_subfigure(gs_in[:, 1])     
contri = lower.add_subfigure(gs_in[0:2, 0])  

#get axes
ax_upper = upper.subplots(1, 3).flatten()  
ax_rel_conpro = rel_conpro.subplots(3, 1).flatten()
ax_contri = contri.subplots(1, 1) 

#################################################################################
#upper part
season_dict = dict(zip([0, 1, 2], ["MAM", "JJA", "SON"]))
title_dict = dict(zip([0, 1, 2], ["(a) MAM", "(b) JJA", "(c) SON"]))

for ax_id, ax in enumerate(ax_upper):


    #------------------------------------
    study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")    
    bbox_adjusted = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Region_Ten.shp")   
    
    region_positions = {}
    region = ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]  #!!! Please make sure this is the same order as the column order in the ES-matrix dataframe.
    
    region_positions['BI'] = (-5, 57)
    region_positions['IP'] = (-5, 39.0)
    region_positions['FR'] = (-2, 47)
    region_positions['ME'] = (11.5, 53)
    region_positions['AL'] = (11, 45.5)
    region_positions['SEA'] = (23, 47)
    region_positions['NEA'] = (28, 57)
    region_positions['SC'] = (17, 68)
    region_positions['WMD'] = (12, 36)
    region_positions['EMD'] = (28, 39)
    
    #------------------------------------
    season = season_dict[ax_id]
    
    #load dependency
    dependency = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Analyze_synchronized_event_pair/Dependency_wr_for_region_pair_{season}_incl_noregime.csv")
    
    #load ensemble dependency
    dependency_ens = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Analyze_synchronized_event_pair/Dependency_ens2000_wr_for_region_pair_{season}_incl_noregime.csv")
    
    #construct network
    #--------------------------------
    G = nx.MultiGraph()
    
    #add nodes
    G.add_nodes_from(region)

    #small changes to region abbreviations
    node_labels = {
    'BI':'BI',
    'IP':'IP',
    'FR':'FR',
    'ME':'CE', #ME->CE
    'AL':'AL',
    'SEA':'SEE',  #SEA->SEE
    'NEA':'NEE', #NEA->NEE
    'SC':'SC',
    'WMD':'WMD',
    'EMD':'EMD'
    }
    nx.set_node_attributes(G, node_labels, "label")
    
    
    #add edges
    for (reg1, reg2) in [(combo[0], combo[1]) for combo in combinations(region, 2)]:
    
        #check observation numbers
        n_obs = dependency.loc[(dependency["reg1"] == reg1) & (dependency["reg2"] == reg2), "n"].item()
    
        if n_obs <= 20:
            continue
            
        else:
            dependency_ens_reg = dependency_ens[(dependency_ens["reg1"] == reg1) & (dependency_ens["reg2"] == reg2)]
    
            n = 0
            rad_dict = dict(zip([0, 1, 2], [0, -0.1, 0.1]))   #curvature for network visualization
            
            for wrname in ["EuBL", "GL", "ScBL", "ScTr", "AR", "AT", "ZO", "no"]:
                
                dp = dependency.loc[(dependency["reg1"] == reg1) & (dependency["reg2"] == reg2), wrname].item()
                sig_thd = dependency_ens_reg[wrname].quantile(0.95)
    
                if dp > sig_thd:
                    
                    #add edge
                    scaling_factor = 3   #adjust edge width here
                    G.add_edge(reg1, reg2, rad = rad_dict[n], color = wr_color_dict[wrname], weight = 1 * scaling_factor)   #equal linewidth
                    n += 1 
    
    #--------------------------------------------------------------------------------------
    #Draw the study area and regionalization
    study_area.boundary.plot(ax = ax, color = '#dddddd', zorder = 0)
    bbox_adjusted.boundary.plot(ax = ax, color='#aaaaaa', zorder = 0)
    
    # Draw edges
    for u, v, key, data in G.edges(keys=True, data=True):
        edge_color = data.get('color')  
        edge_weight = data.get('weight')  
        rad = data.get('rad')  
    
        # Draw the individual edge with attributes
        nx.draw_networkx_edges(
            G,
            pos = region_positions,
            ax = ax,
            edgelist=[(u, v)],  # Plot a single edge at a time
            edge_color=[edge_color],
            style = '-', 
            width=edge_weight,
            connectionstyle=f'arc3,rad={rad}'  # Add curvature for multi-edges
        )
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos = region_positions, ax=ax, node_color="#555555", node_size=1100).set_zorder(1)
    
    # Draw labels
    nx.draw_networkx_labels(G, pos = region_positions, ax=ax, labels = node_labels, font_color = "white", font_size=13, font_weight = "bold")
    
    
    # Subpanel title
    ax.text(0.02, 0.95, title_dict[ax_id], fontsize = 18, ha = 'left', va = 'bottom', transform = ax.transAxes)
    
    # turn the axis on
    ax.set_axis_on()
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.set_xlabel("")
    ax.set_ylabel("")


#title
upper.suptitle("Pair-wise synchronicity", fontsize = 20, x = 0, y = 0.99, ha = 'left', va = 'bottom')

#################################################################################
#dependency

for dfkey, ax in zip(["MAM", "JJA", "SON"], ax_rel_conpro):

    #load dependency table, leave out reginal synchronicity values, leave out no regimes
    df = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Weather_Regime_Dependency/Weather_Regime_Dependency_five_levels_{dfkey}_incl_noregime.csv")
    sample_size = list(df["n"])


    #melt
    df_melt = df.melt(id_vars = 'synlevel',
                      value_vars = ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL", "no"],
                      var_name = "WR",
                      value_name = "dependency")

    ax.axvline(x = 1.0, color = "#909090", linestyle = "--")
    sns.scatterplot(x='dependency', y='synlevel', hue='WR', data=df_melt, palette=wr_color_dict, marker="x", lw = 2, s = 80, ax = ax)
    title = "(e) MAM" if dfkey == "MAM" else "(f) JJA" if dfkey == "JJA" else "(g) SON"
    ax.text(0.83, 0.78, title, va = 'bottom', ha = 'left', fontsize = 18, transform = ax.transAxes)

    ax.set_ylabel('')
    ax.set_yticks([0, 1, 2, 3, 4])
    ax.set_yticklabels([f'No\nn={sample_size[0]}', f'Regional\nn={sample_size[1]}', f'Low\nn={sample_size[2]}', f'Medium\nn={sample_size[3]}', f'High\nn={sample_size[4]}'], fontsize=12)
    ax.set_xlim(-0.05, 3.5)
    ax.legend().set_visible(False)
    
    if dfkey == "SON":
        ax.set_xlabel("Relative conditional probability (-)", fontsize = 15)
        ax.set_xticks([0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5])
        ax.set_xticklabels([0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5], fontsize = 13)
    else:
        ax.set_xlabel("")
        ax.set_xticks([])
        ax.set_xticklabels([])

    
    #add lines
    for wrt in ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL"]:

        if df[wrt].is_monotonic_increasing or df[wrt].is_monotonic_decreasing:
            val = list(df[wrt])
            ax.plot([val[0], val[1]], [0, 1], color = wr_color_dict[wrt], linestyle = '-', linewidth = 1)
            ax.plot([val[1], val[2]], [1, 2], color = wr_color_dict[wrt], linestyle = '-', linewidth = 1)
            ax.plot([val[2], val[3]], [2, 3], color = wr_color_dict[wrt], linestyle = '-', linewidth = 1)
            ax.plot([val[3], val[4]], [3, 4], color = wr_color_dict[wrt], linestyle = '-', linewidth = 1)

rel_conpro.suptitle("Continental synchronicity", fontsize = 20, x = 0, y = 1, ha = 'left', va = 'bottom')

#################################################################################
#contribution
MAM_regimes = ["EuBL", "ZO", "ScBL", "ScTr"]
MAM_count = [9, 4, 3, 1] #17
JJA_regimes = ["AR", "ScBL", "AT", "EuBL", "ZO", "no"]
JJA_count = [5, 4, 3, 1, 1, 2] #16
SON_regimes = ["ZO", "ScTr", "EuBL"]
SON_count = [2, 2, 1] #5

regimes = ["EuBL", "ScBL", "ZO", "AR", "ScTr", "AT", "no", "GL"]
frac = [11/38, 7/38, 7/38, 5/38, 3/38, 3/38, 2/38, 0]

ax_contri.bar(regimes, frac, color = [wr_color_dict[regime] for regime in regimes], edgecolor = "black", width = 0.5, linewidth = 2)
ax_contri.set_ylabel("Fraction of contribution (-)", fontsize = 15)
ax_contri.tick_params(axis = "x", labelsize = 16.5)
ax_contri.tick_params(axis = "y", labelsize = 12)
ax_contri.text(0.9, 0.9, "(d)", ha = 'left', va = 'bottom', fontsize = 18, transform  = ax_contri.transAxes)

contri.suptitle("Weather regime contribution", fontsize = 20, x = 0, y = 1, ha = 'left', va = 'bottom')
#################################################################################
#legend
# Anticyclonic

mksize = 10  #markersize
hdlen = 3  #handellength
lw = 1.5  #linewidth
mkew = 2  #markeredgewidth


EuBL = mlines.Line2D([], [], color=wr_color_dict['EuBL'], linewidth=lw, label='EuBL', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
ScBL = mlines.Line2D([], [], color=wr_color_dict['ScBL'], linewidth=lw, label='ScBL', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
AR = mlines.Line2D([], [], color=wr_color_dict['AR'], linewidth=lw, label='AR', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
GL = mlines.Line2D([], [], color=wr_color_dict['GL'], linewidth=lw, label='GL', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
ScTr = mlines.Line2D([], [], color=wr_color_dict['ScTr'], linewidth=lw, label='ScTr', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
AT = mlines.Line2D([], [], color=wr_color_dict['AT'], linewidth=lw, label='AT', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
ZO = mlines.Line2D([], [], color=wr_color_dict['ZO'], linewidth=lw, label='ZO', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
no = mlines.Line2D([], [], color=wr_color_dict['no'], linewidth=lw, label='no', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)

wr.legend(handles = [EuBL, ScBL, AR, GL, ScTr, AT, ZO, no], ncol = 4, title = "Weather regime", bbox_to_anchor = (0.25, 0.1), frameon = False, handlelength = hdlen, title_fontsize = 18, loc = "center", fontsize = 17)

#---------------------------------
#save
#wr.savefig("/home/lixinh/SynFire_Scripts/WP1/Figures/v5_4_weather_regime.png", dpi = 600, bbox_inches = "tight")